In [22]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)
import composition_stats as cs
import math
from scipy.stats import chi2
from scipy import stats

In [23]:
def low_counts_rm(data_offset, thres = 0.01):
    col_sum = data_offset.sum(axis = 0)
    ttl_sum = col_sum.sum()
    keep_otu = col_sum[(col_sum * 100 / ttl_sum) > thres].index
    data_offset_v2 = data_offset[keep_otu]
    data_offset_v2 = data_offset_v2.astype(float)
    return data_offset_v2, keep_otu

def label_data(rawdata, metadata):
    taxa_lst = rawdata['FeatureID']
    rawdata_v1 = rawdata.T.drop(['FeatureID'])
    rawdata_v1.columns = list(taxa_lst)
    rawdata_v1 = rawdata_v1.drop(['Unnamed: 0']).reset_index()
    return rawdata_v1.merge(metadata[['Unnamed: 0', 'disease']], how = 'left', left_on = ['index'],  right_on = ['Unnamed: 0']).drop(['index', 'Unnamed: 0'], axis = 1)


def split_processing_single(data, thres = 0.01):
    data_hea = data[data['Disease'] == 'Healthy'].reset_index().drop(['index'], axis = 1)
    print(data_hea.shape)
    data_t2d = data[data['Disease'] == 'T2D'].reset_index().drop(['index'], axis = 1)
    print(data_t2d.shape)
    data_hea_offset = data_hea.drop(['Disease', 'sample_id'], axis = 1) # + data_hea.min()[1:-1].min()
    data_t2d_offset = data_t2d.drop(['Disease', 'sample_id'], axis = 1) # + data_t2d.min()[1:-1].min()
    data_heainfo = low_counts_rm(data_hea_offset, thres)
    data_t2dinfo = low_counts_rm(data_t2d_offset, thres)
    Union_taxa = list(set(data_heainfo[1]) | set(data_t2dinfo[1]))
    data_v1 = data[Union_taxa]
    # data_v1 += 1
    data_v1 = data_v1.astype('float64')
    # data_v1['BioProject'] = data['BioProject']
    data_v1['Disease'] = data['Disease']
    data_v1['sample_id'] = data['sample_id']
    return data_v1

def clr_trans_modify(data, last_ver_data, offset = 1):
    if 'disease' in list(data.columns):
        return 'drop disease column at first'
    data_v1 = pd.DataFrame(cs.clr(cs.closure(data + offset)))
    data_v1.columns = list(data.columns)
    data_v1['disease'] = last_ver_data['Disease']
    # data_v1['sample_id'] = last_ver_data['sample_id']
    # data_v1['study'] = study_name
    return data_v1
    
def clr_trans(data, last_ver_data, study_name = None):
    if 'disease' in list(data.columns):
        return 'drop disease column at first'
    data_v1 = pd.DataFrame(cs.clr(cs.closure(data + 1)))
    data_v1.columns = list(data.columns)
    data_v1['disease'] = last_ver_data['disease']
    if study_name is not None:
        data_v1['study'] = study_name
    return data_v1

In [24]:
sample_md = pd.read_excel(r"C:\Users\edwar\Desktop\Melbourne\research_project\HGMA_data\sample_metadata.xlsx")
abund_data = pd.read_csv(r"C:\Users\edwar\Desktop\Melbourne\research_project\HGMA_data\vect_atlas.csv\vect_atlas.csv")
corr_taxa = pd.read_csv(r"C:\Users\edwar\Desktop\Melbourne\research_project\HGMA_data\corresponding_taxa.csv")

corr_taxa = corr_taxa[~corr_taxa['name'].str.contains('unclassified')].reset_index(drop = True)
multi_taxa = corr_taxa[corr_taxa['name'].str.contains(r"\s\d$")].reset_index(drop = True)
multi_taxa['name'] = multi_taxa['name'].apply(lambda x : x[:-2])
multi_taxa_lst = list(multi_taxa['name'].unique())

In [25]:
dataset_study = "PRJEB1786"

In [26]:
study2_metadata = sample_md[sample_md['BioProject'] == dataset_study].reset_index(drop = True)    
metadata = study2_metadata
metadata = metadata[metadata['Disease'].isin(['T2D', 'Healthy', 'NGT'])].reset_index(drop = True)
sp_id = list(metadata['sample.ID'])
rawdata = abund_data[sp_id + ['Unnamed: 0']]
rawdata = rawdata.merge(corr_taxa, how = 'left', left_on = ['Unnamed: 0'], right_on = ['id']).drop(['Unnamed: 0', 'id'], axis = 1)
rawdata = rawdata[rawdata['name'].isna() == False].reset_index(drop = True)


In [27]:
taxa_lst = list(rawdata['name'])
rawdata_v1 = rawdata.drop(['name'], axis = 1).T
rawdata_v1.columns = taxa_lst
rawdata_v1 = rawdata_v1.reset_index().rename(columns = {'index':'sample_id'})
rawdata_v1 = rawdata_v1.merge(metadata[['sample.ID', 'Disease']], how = 'left', left_on = ['sample_id'], right_on = ['sample.ID']).drop(['sample.ID'], axis = 1)

rawdata_v1 = rawdata_v1.rename(columns = {'Blautia coccoides == Blautia producta':'Blautia producta'})

In [28]:
rawdata_v1.shape

(96, 840)

In [30]:
# filtering out the inconsistent naming of taxa
def check_main_species(lst):
    contain_main_species = False
    integers = [str(i) for i in range(1, 10)]
    for sub in lst:
        if sub[-1] not in integers:
            contain_main_species = True
            break
    # True means it contains main species, False means all are subspecies which should sum them up in the next step
    return contain_main_species

for taxa in multi_taxa_lst:
    if 'subtype' not in taxa.lower():
        cols = list(rawdata_v1.columns)
        rawdata_v1[taxa] = 0
        alltaxa_columns = [col for col in cols if taxa in col]
        if len(alltaxa_columns) == 1:
            rawdata_v1[taxa] = rawdata_v1[alltaxa_columns[0]]

        elif check_main_species(alltaxa_columns):
            alltaxa_columns.remove(taxa)
            rawdata_v1 = rawdata_v1.drop(alltaxa_columns, axis = 1)

        else:
            rawdata_v1[taxa] = rawdata_v1[alltaxa_columns].sum(axis = 1)
            rawdata_v1 = rawdata_v1.drop(alltaxa_columns, axis = 1)
     

In [ ]:
# rename the column names
rawdata_v1 = rawdata_v1.rename(
    columns={'Lachnospiraceae bacterium hRUG904 / Clostridiales bacterium RUG495 / [Clostridium] sp. CAG:127':'[Clostridium] sp. CAG:127', 
             'Firmicutes bacterium CAG:65 / [Clostridium] sp. 2789STDY5608883':'Firmicutes bacterium CAG:65', 
             'Lachnospiraceae bacterium CIM:MAG 844 / Firmicutes bacterium CAG:95':'Firmicutes bacterium CAG:95', 
             '[Ruminococcus] sp. CAG:17 / Blautia sp. 2789STDY5608880':'[Ruminococcus] sp. CAG:17', 
             'Roseburia sp. CAG:100 / Lachnospiraceae bacterium CIM:MAG 54':'Roseburia sp. CAG:100', 
             'Clostridiales bacterium UBA7273 / Firmicutes bacterium CAG:240':'Firmicutes bacterium CAG:240', 
             'Firmicutes bacterium CAG:24 / [Clostridium] sp. 2789STDY5834869 & 2789STDY5834922':'Firmicutes bacterium CAG:24',
             'Coprococcus sp. 2789STDY5608819 / [Clostridium] sp. CAG:264':'[Clostridium] sp. CAG:264',
             '[Ruminococcus] sp. 2789STDY5608794 & sp. 2789STDY5834890 / Firmicutes bacterium CAG:56':'Firmicutes bacterium CAG:56',
             'Lachnospiraceae bacterium TF01-11 / [Clostridium] sp. CAG:122':'[Clostridium] sp. CAG:122',
             '[Acinetobacter] sp. N54.MGS-139 / Proteobacteria bacterium CAG:139':'Proteobacteria bacterium CAG:139',
             '[Ruminococcus] sp. CAG:60 / Blautia sp. 2789STDY5608836':'[Ruminococcus] sp. CAG:60', 
             '[Eubacterium] sp. CAG:251 / [Clostridium] sp. A254.MGS-251':'[Eubacterium] sp. CAG:251',
             '[Clostridium] sp. CAG:217 / Clostridiales bacterium CIM:MAG 317_1':'[Clostridium] sp. CAG:217', 
             'Lachnoclostridium sp. SNUG30386 / [Clostridium] sp. CAG:43':'[Clostridium] sp. CAG:43', 
             '[Ruminococcaceae] bacterium D16 / Clostridiales bacterium Choco116':'[Ruminococcaceae] bacterium D16', 
             'Firmicutes bacterium CAG:41 / [Clostridium] sp. 2789STDY5834935 & sp. 2789STDY5608853':'Firmicutes bacterium CAG:41', 
             '[Clostridiaceae] bacterium CIM:MAG 755 / [Clostridium] sp. CAG:230':'[Clostridium] sp. CAG:230', 
             'Firmicutes bacterium CAG:212 / [Clostridium] sp. 2789STDY5834871':'Firmicutes bacterium CAG:212', 
             'Bacteroidales bacterium H2 / [Prevotella] sp. CAG:251':'[Prevotella] sp. CAG:251', 
             'Ruminococcaceae bacterium UBA1821 / [Clostridium] sp. CAG:678':'[Clostridium] sp. CAG:678', 
             'Oscillibacter sp. ER4 / Firmicutes bacterium CAG:129_59_24':'Oscillibacter sp. ER4', 
             'Clostridiales bacterium RUG303 & RUG439 & RUG453 / Firmicutes bacterium CAG:129':'Firmicutes bacterium CAG:129', 
             'Candidatus Methanomethylophilus alvus / Methanoculleus sp. CAG:1088':'Methanoculleus sp. CAG:1088', 
             '[Clostridiaceae] bacterium CIM:MAG 987 / [Clostridium] sp. CAG:533':'[Clostridium] sp. CAG:533', 
             '[Firmicutes] bacterium CIM:MAG 721 / [Bacillus] sp. CAG:988':'[Bacillus] sp. CAG:988'
            }
)

In [16]:
drop_cols = [taxa for taxa in list(rawdata_v1.columns)[1:-1] if (len(rawdata_v1[taxa].unique()) == 1) & (taxa != 'Disease')]
rawdata_v2 = split_processing_single(rawdata_v1, thres = 0.0001); print(rawdata_v2.shape)

(0, 824)
(53, 824)
(96, 599)


,Enterococcus faecium,[Anaerotruncus] sp. CAG:390,Coprobacter secundus == Gabonia massiliensis,Ruminococcus sp. CAG:624 & UBA6407 & UBA4697,Bacteroidales bacterium H3,Bacteroides nordii,Prevotella sp. P4-65 & P4-76 & P5-60 & P5-119 & P5-125 & CAG:592,Streptococcus sanguinis,Faecalicoccus pleomorphus,Lactobacillus delbrueckii,[Bacilli] bacterium Phil2,Clostridiales bacterium CIM:MAG 442,Faecalibacterium prausnitzii,[Clostridium] sp. 42_12 & CAG:75,Megasphaera elsdenii,Bacteroides plebeius,Merdimonas faecis,[Firmicutes] bacterium CAG:321,Collinsella tanakaei,[Bacteroides] sp. CAG:1060,Clostridiales bacterium RUG303 & RUG439 & RUG453 / Firmicutes bacterium CAG:129,Ruminococcaceae bacterium D5,Catenibacterium sp. CAG:290,Clostridiales bacterium CIM:MAG 912,Streptococcus infantis 2,...,Pseudoflavonifractor sp. An184,Clostridiales bacterium CIM:MAG 1534,Bacteroides salyersiae,[Firmicutes] bacterium CAG:313,Streptococcus anginosus,[Clostridium] spiroforme,[Clostridium] sp. 28_12 & CAG:356,Mitsuokella multacida,Hungatella hathewayi 1,Opitutae bacterium UBA4654,Ruminococcus bicirculans,Firmicutes bacterium CAG:475,[Clostridium] sp. CAG:798,Ruminococcaceae bacterium CIM:MAG 577,Lachnoclostridium sp. UBA2905,Streptococcus vestibularis,Muribaculaceae bacterium sp2,Ruminococcus sp. CAG:177,Coprococcus sp. CAG:782,[Eubacterium] sp. CAG:192,[Acholeplasma] sp. CAG:878,[Clostridiaceae] bacterium CIM:MAG 755 / [Clostridium] sp. CAG:230,Bacteroides sp. CAG:1076,Disease,sample_id
0,0.0,0.0,0.000000e+00,0.0,0.0,1.815240e-08,0.0,0.000000e+00,0.0,1.586596e-08,0.000000,0.000000e+00,0.000002,9.668760e-07,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,2.336161e-08,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.0,0.0,0.0,0.000000e+00,0.000000e+00,3.200100e-07,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,1.112756e-08,0.0,0.000000e+00,0.0,9.310790e-07,0.000000e+00,0.000000e+00,0.0,T2D,ERS235503
1,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00,0.0,8.377945e-09,0.0,0.000000e+00,0.000000,0.000000e+00,0.000004,2.599685e-07,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,2.497578e-08,...,0.0,0.0,0.0,0.000000e+00,5.707419e-08,3.569029e-07,0.000000e+00,0.0,4.232279e-08,0.0,2.691079e-07,0.000000e+00,0.0,0.000000e+00,0.0,8.176370e-09,0.0,0.000000e+00,0.0,7.529871e-07,0.000000e+00,0.000000e+00,0.0,T2D,ERS235504
2,0.0,0.0,8.801301e-09,0.0,0.0,9.149987e-09,0.0,3.879301e-08,0.0,0.000000e+00,0.000000,0.000000e+00,0.000006,2.455818e-06,0.0,1.013057e-06,0.000000e+00,0.000000e+00,0.0,0.0,1.104702e-06,1.684557e-08,0.000000e+00,0.0,0.000000e+00,...,0.0,0.0,0.0,7.379294e-07,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,2.485089e-07,1.775043e-07,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.0,7.260625e-07,3.580752e-07,5.413590e-07,0.0,T2D,ERS235508
3,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.000000e+00,0.0,3.675772e-07,0.000002,1.045358e-08,0.000001,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,1.426490e-08,0.000000e+00,0.000000e+00,0.0,0.000000e+00,...,0.0,0.0,0.0,0.000000e+00,0.000000e+00,2.516465e-08,0.000000e+00,0.0,1.569281e-08,0.0,8.891228e-08,0.000000e+00,0.0,0.000000e+00,0.0,6.017982e-08,0.0,1.156016e-05,0.0,0.000000e+00,0.000000e+00,1.321290e-08,0.0,NGT,ERS235511
4,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.000000,0.000000e+00,0.000014,2.709897e-08,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,4.746966e-06,0.000000e+00,0.000000e+00,0.0,1.815542e-08,...,0.0,0.0,0.0,0.000000e+00,0.000000e+00,3.235561e-07,0.000000e+00,0.0,5.054259e-09,0.0,3.468759e-07,0.000000e+00,0.0,0.000000e+00,0.0,7.032044e-08,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,T2D,ERS235513
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,0.0,0.0,0.000000e+00,0.0